# Git Tutorial - Part 4: Conflict Resolution
## Master the Art of Resolving Merge Conflicts

## 9. Understanding Conflicts

### What is a Merge Conflict?
A conflict occurs when Git **cannot automatically** merge changes because:
- **Same lines** were modified differently in two branches
- A file was **deleted** in one branch but **modified** in another
- **Binary files** were changed in both branches

### When Do Conflicts Happen?
1. **Merging branches**: `git merge feature`
2. **Rebasing**: `git rebase main`
3. **Cherry-picking**: `git cherry-pick abc123`
4. **Pulling**: `git pull origin main`

### Don't Panic!
- Conflicts are **normal** in collaborative development
- They're **not errors** - just Git asking for help
- With practice, they're **easy to resolve**

## 10. Conflict Markers Explained

### Anatomy of a Conflict

When a conflict occurs, Git marks the file like this:

```python
def calculate_total(price, quantity):
<<<<<<< HEAD
    # Current branch (yours)
    total = price * quantity * 1.1  # Added 10% tax
    return round(total, 2)
=======
    # Incoming branch (theirs)
    discount = 0.05
    total = price * quantity * (1 - discount)
    return total
>>>>>>> feature/discount
```

### Marker Meanings:
- `<<<<<<< HEAD`: Start of YOUR changes (current branch)
- `=======`: Separator between changes
- `>>>>>>> branch-name`: End of THEIR changes (incoming branch)

### Your Job:
1. **Decide** which version to keep (or combine both)
2. **Remove** conflict markers
3. **Test** the code
4. **Stage** the resolved file
5. **Complete** the merge

## 11. Detecting Conflicts

In [ ]:
# Check if merge is in progress
# SCENARIO: You started a merge, got interrupted, forgot where you were
# WHY: Know current state
# OUTPUT: Shows "You are in the middle of a merge"
!git status

# List conflicted files only
# SCENARIO: Many files changed, only care about conflicts
# WHY: Focus on what needs fixing
# OUTPUT: Just the files with conflicts
!git diff --name-only --diff-filter=U

# See conflict details
# SCENARIO: Want to see what's conflicting
# WHY: Understand the conflict before opening files
!git diff

# Check merge status
# SCENARIO: How many files left to resolve?
# OUTPUT: Shows unmerged paths
!git status

## 12. Resolving Conflicts - Step by Step

### Method 1: Manual Resolution (Most Common)

In [ ]:
# STEP-BY-STEP WORKFLOW:

# 1. Attempt merge
!git merge feature/new-feature
# Output: "CONFLICT (content): Merge conflict in app.py"

# 2. Check which files have conflicts
!git status
# Shows: "both modified: app.py"

# 3. Open conflicted file in editor
# Find conflict markers (<<<<<<, =======, >>>>>>>)
# Manually edit to keep desired changes
# Remove ALL conflict markers

# 4. After editing, stage the resolved file
!git add app.py

# 5. Check if all conflicts resolved
!git status
# Should show: "All conflicts fixed but you are still merging"

# 6. Complete the merge
!git commit
# Git opens editor with default merge message
# Or specify message:
!git commit -m "Merge feature/new-feature - resolved conflicts in app.py"

### Method 2: Accept One Side Completely

In [ ]:
# Accept THEIR version (incoming branch)
# SCENARIO: Their changes are correct, yours are wrong
# WHY: Quick resolution when one side is clearly right
# WHEN: You made experimental changes, theirs is the fix
!git checkout --theirs path/to/file.py
!git add path/to/file.py

# Accept OUR version (current branch)
# SCENARIO: Your changes are correct, ignore theirs
# WHY: Keep your version, discard incoming
# WHEN: You know your version is the right one
!git checkout --ours path/to/file.py
!git add path/to/file.py

# Accept theirs for ALL conflicts
# SCENARIO: Mass conflict, their branch is authoritative
# CAUTION: Discards ALL your changes!
!git checkout --theirs .
!git add .

# Accept ours for ALL conflicts
# SCENARIO: Your branch is authoritative
!git checkout --ours .
!git add .

### Method 3: Using Merge Tools

In [ ]:
# Configure merge tool (one-time setup)
# SCENARIO: Want visual tool for conflicts
# WHY: Easier than manual editing
# OPTIONS: vimdiff, meld, kdiff3, p4merge, Beyond Compare
!git config --global merge.tool meld

# Launch merge tool
# SCENARIO: Conflict occurred, want GUI help
# WHY: Visual 3-way diff is clearer
# WHAT HAPPENS: Opens configured tool for each conflict
!git mergetool

# Use specific tool (override config)
# SCENARIO: Try different tool for this conflict
!git mergetool --tool=vimdiff

# After resolving in tool:
# - Tool saves resolved file
# - Git automatically stages it
# - Complete merge with commit
!git commit

## 13. Advanced Conflict Resolution

### 13.1 Three-Way Diff

In [ ]:
# View three versions during conflict
# SCENARIO: Need to see common ancestor to understand conflict
# WHY: Context helps make better decisions

# Stage 1: Common ancestor (base)
!git show :1:filename.py

# Stage 2: Our version (HEAD)
!git show :2:filename.py

# Stage 3: Their version (merging branch)
!git show :3:filename.py

# REAL-WORLD USE:
# Base: total = price * quantity
# Ours: total = price * quantity * 1.1  # Added tax
# Theirs: total = price * quantity * 0.95  # Added discount
# Resolution: total = price * quantity * 1.1 * 0.95  # Both!

### 13.2 Merge Strategies

In [ ]:
# Merge with strategy: prefer ours
# SCENARIO: Merge but auto-resolve conflicts with our version
# WHY: Bulk merge where your branch is authoritative
# WHEN: Merging old branch, keeping current work
!git merge -X ours feature/old-branch

# Merge with strategy: prefer theirs
# SCENARIO: Accept their changes for conflicts
# WHY: Their branch has priority
# WHEN: Merging from main into feature
!git merge -X theirs main

# Ignore whitespace conflicts
# SCENARIO: Conflicts due to formatting changes
# WHY: Focus on real code changes
# WHEN: Different editors/formatters used
!git merge -X ignore-space-change feature/reformatted
!git merge -X ignore-all-space feature/reformatted

### 13.3 Rerere (Reuse Recorded Resolution)

In [ ]:
# Enable rerere (one-time setup)
# SCENARIO: Repeatedly resolving same conflicts
# WHY: Git remembers how you resolved conflicts
# WHEN: Long-lived branches, frequent rebasing
# BENEFIT: Auto-applies previous resolutions
!git config --global rerere.enabled true

# HOW IT WORKS:
# 1. First time: You resolve conflict manually
# 2. Git records your resolution
# 3. Next time same conflict occurs: Auto-resolved!

# View rerere cache
!git rerere status

# Clear rerere cache
# SCENARIO: Recorded resolution was wrong
!git rerere forget path/to/file.py

# REAL-WORLD EXAMPLE:
# - Rebase feature branch on main weekly
# - Same conflicts appear each time
# - With rerere: Conflicts auto-resolved!
# - Saves hours of repetitive work

## 14. Aborting and Recovering

### 14.1 Aborting Merge

In [ ]:
# Abort merge and return to pre-merge state
# SCENARIO: Too many conflicts, wrong approach
# WHY: Start over with different strategy
# WHEN: Realized you should rebase instead
# RESULT: Returns to state before 'git merge'
!git merge --abort

# TYPICAL WORKFLOW:
# git merge feature
# # See 100 conflicts
# git merge --abort
# # Try different approach:
# git rebase feature

### 14.2 Recovering from Bad Merge

In [ ]:
# Undo merge commit (before push)
# SCENARIO: Completed merge but it broke everything
# WHY: Merge was wrong, need to redo
# WHEN: Immediately after bad merge
!git reset --hard HEAD~1

# Undo merge commit (after push)
# SCENARIO: Pushed bad merge, can't use reset
# WHY: Preserve history, safe for shared branches
# WHEN: Bad merge already pushed to remote
!git revert -m 1 HEAD
# -m 1 means: keep first parent (usually main branch)

# Find merge commit to revert
!git log --merges --oneline
!git revert -m 1 abc1234

## 15. Rebase Conflicts

### Rebase vs Merge Conflicts
- **Merge**: One conflict resolution session
- **Rebase**: Potentially multiple conflict sessions (one per commit)

### 15.1 Resolving Rebase Conflicts

In [ ]:
# Start rebase
!git rebase main
# Conflict occurs

# 1. Check status
!git status
# Shows: "rebase in progress"

# 2. Resolve conflicts (same as merge)
# Edit files, remove markers

# 3. Stage resolved files
!git add resolved-file.py

# 4. Continue rebase (NOT commit!)
# SCENARIO: Resolved current commit's conflicts
# WHY: Move to next commit in rebase
# NOTE: Don't use 'git commit' during rebase!
!git rebase --continue

# If more commits have conflicts, repeat steps 2-4

# Skip current commit
# SCENARIO: Current commit is no longer needed
# WHY: Changes already in target branch
# WHEN: Commit became redundant
!git rebase --skip

# Abort rebase
# SCENARIO: Too many conflicts, wrong approach
# WHY: Return to pre-rebase state
!git rebase --abort

## 16. Conflict Prevention

### Best Practices to Minimize Conflicts:

1. **Pull frequently**: Stay in sync with main branch
2. **Small commits**: Easier to resolve conflicts
3. **Communicate**: Coordinate with team on file changes
4. **Feature branches**: Isolate work
5. **Rebase regularly**: Keep feature branch updated
6. **Code review**: Catch conflicts early

In [ ]:
# Keep feature branch updated (prevents conflicts)
# SCENARIO: Working on long-lived feature branch
# WHY: Smaller, incremental conflict resolution
# WHEN: Daily or weekly

# Method 1: Merge main into feature
!git switch feature/my-feature
!git merge main

# Method 2: Rebase feature on main (cleaner)
!git switch feature/my-feature
!git rebase main

# Check for potential conflicts before merging
# SCENARIO: Want to preview conflicts
# WHY: Plan resolution strategy
!git merge --no-commit --no-ff feature/test
!git diff --check  # Check for conflicts
!git merge --abort  # Abort test merge

## 17. Common Conflict Scenarios

### Scenario 1: Same Line Modified
**Situation**: Two developers edited same line
**Resolution**: Combine changes or choose best version

### Scenario 2: File Deleted vs Modified
**Situation**: One branch deleted file, other modified it
**Resolution**: Decide if file should exist with modifications or be deleted

### Scenario 3: Rename Conflicts
**Situation**: File renamed in one branch, modified in another
**Resolution**: Git usually handles this, but may need manual help

### Scenario 4: Binary File Conflicts
**Situation**: Images, PDFs, etc. modified in both branches
**Resolution**: Choose one version (can't merge binary files)
```bash
git checkout --ours image.png
# or
git checkout --theirs image.png
```